# Aria Scalability & Performance Tuning Guide

**Advanced guide for scaling architecture, capacity planning, and performance optimization at enterprise scale.**

Learn how to scale Aria to handle millions of users and requests.


## Scalability Architecture

### Horizontal Scaling Tiers

```
Tier 1: Single Server (Development)
├─ Single machine with Python + SQLite
└─ Capacity: 10 concurrent users

Tier 2: Application + Database (Small Production)
├─ Separate app server and PostgreSQL
├─ Single instance
└─ Capacity: 100-500 concurrent users

Tier 3: Distributed Application (Medium Production)
├─ Load balancer
├─ Multiple app instances
├─ Managed PostgreSQL (read replicas)
├─ Redis cache
└─ Capacity: 1,000-10,000 concurrent users

Tier 4: Multi-Region (Large Production)
├─ Regional load balancers
├─ Auto-scaling groups by region
├─ Cross-region DB replication
├─ CDN for static assets
├─ Distributed caching
└─ Capacity: 10,000+ concurrent users
```

### Load Testing

```python
# scripts/load_test_aria.py
import locust
import random
from locust import HttpUser, task, between

class AriaUser(HttpUser):
    wait_time = between(1, 3)

    def on_start(self):
        """Setup: login once per user."""
        self.auth_token = self.login()

    def login(self):
        """Simulate login."""
        response = self.client.post(
            "/api/auth/login",
            json={"email": f"user{random.randint(1, 1000)}@example.com", "password": "test"}
        )
        return response.json().get("token")

    @task(70)
    def chat_message(self):
        """70% of traffic: chat messages."""
        headers = {"Authorization": f"Bearer {self.auth_token}"}
        self.client.post(
            "/api/chat/completions",
            json={"messages": [{"role": "user", "content": "Hello"}]},
            headers=headers
        )

    @task(15)
    def get_history(self):
        """15% of traffic: get chat history."""
        headers = {"Authorization": f"Bearer {self.auth_token}"}
        self.client.get(
            "/api/chat/history",
            headers=headers
        )

    @task(10)
    def character_action(self):
        """10% of traffic: Aria character actions."""
        headers = {"Authorization": f"Bearer {self.auth_token}"}
        self.client.post(
            "/api/aria/command",
            json={"command": "move left"},
            headers=headers
        )

    @task(5)
    def get_status(self):
        """5% of traffic: status check."""
        self.client.get("/api/ai/status")

# Run: locust -f scripts/load_test_aria.py --host=http://localhost:7071 --users=1000 --spawn-rate=10
```

### Capacity Planning

```python
# Estimate infrastructure requirements
class CapacityPlanner:
    def __init__(self):
        # Baseline metrics
        self.requests_per_user_per_hour = 20  # Average
        self.avg_response_time_ms = 250
        self.peak_factor = 3  # Peak is 3x average

    def plan_for_users(self, daily_active_users: int):
        """Calculate infrastructure needs."""
        peak_concurrent = (daily_active_users * self.peak_factor) / 24
        peak_rps = (peak_concurrent * self.requests_per_user_per_hour) / 3600

        return {
            "peak_concurrent_users": int(peak_concurrent),
            "peak_requests_per_second": int(peak_rps),
            "db_connections_needed": int(peak_rps * 0.1),  # 10% of peak RPS
            "cache_memory_gb": int(peak_concurrent * 0.1),  # 100MB per 1000 users
            "api_instances": max(2, int(peak_rps / 100)),  # 100 RPS per instance
            "estimated_monthly_cost_usd": self.estimate_cost(peak_concurrent, peak_rps)
        }

    def estimate_cost(self, concurrent_users: int, rps: int) -> float:
        """Estimate monthly infrastructure cost."""
        # Pricing (example for Azure)
        api_cost = max(2, int(rps / 100)) * 73  # $73/month per Standard instance
        db_cost = 500  # Managed PostgreSQL baseline
        cache_cost = 100  # Redis cache
        bandwidth_cost = rps * 3600 * 24 * 30 * 0.0001  # Data transfer

        return api_cost + db_cost + cache_cost + bandwidth_cost

planner = CapacityPlanner()
plan = planner.plan_for_users(daily_active_users=100_000)
print(f"Infrastructure for 100K DAU:")
print(f"  Peak concurrent: {plan['peak_concurrent_users']}")
print(f"  Peak RPS: {plan['peak_requests_per_second']}")
print(f"  API instances: {plan['api_instances']}")
print(f"  Monthly cost: ${plan['estimated_monthly_cost_usd']:,.0f}")
```

---

## Database Performance Tuning

### Query Performance Analysis

```sql
-- Identify slow queries
SELECT query, calls, mean_time, max_time, total_time
FROM pg_stat_statements
WHERE mean_time > 100  -- > 100ms average
ORDER BY total_time DESC
LIMIT 20;

-- Analyze expensive query
EXPLAIN ANALYZE
SELECT u.username, COUNT(m.id) as messages
FROM users u
LEFT JOIN chat_messages m ON u.id = m.user_id
WHERE u.created_at > NOW() - INTERVAL '30 days'
GROUP BY u.id;

-- Create indexes for fast lookups
CREATE INDEX CONCURRENTLY idx_users_created ON users(created_at DESC);
CREATE INDEX CONCURRENTLY idx_messages_user_date ON chat_messages(user_id, created_at DESC);
```

### Connection Pool Optimization

```python
# Tune database connection pool
from sqlalchemy import create_engine
from sqlalchemy.pool import QueuePool

engine = create_engine(
    DATABASE_URL,
    poolclass=QueuePool,
    pool_size=20,              # Connections to keep in pool
    max_overflow=10,           # Additional connections if needed
    pool_recycle=3600,         # Recycle connections hourly
    pool_pre_ping=True,        # Verify connections before use
    echo_pool=True             # Log pool events for debugging
)

# Monitor pool
@app.route("/api/debug/pool-stats")
def pool_stats():
    pool = engine.pool
    return {
        "size": pool.size(),
        "checked_out": pool.checkedout(),
        "overflow": pool.overflow(),
        "total": pool.size() + pool.overflow()
    }
```

### Partitioning Large Tables

```sql
-- Partition chat_messages by date
CREATE TABLE chat_messages_2024 PARTITION OF chat_messages
    FOR VALUES FROM ('2024-01-01') TO ('2025-01-01');

-- Create partitions for each quarter
CREATE TABLE chat_messages_2024_q1 PARTITION OF chat_messages_2024
    FOR VALUES FROM ('2024-01-01') TO ('2024-04-01');

CREATE TABLE chat_messages_2024_q2 PARTITION OF chat_messages_2024
    FOR VALUES FROM ('2024-04-01') TO ('2024-07-01');

-- Archive old partitions to cold storage
ALTER TABLE chat_messages_2024_q1 SET TABLESPACE archive_tablespace;
```

---

## Application Performance Optimization

### Async Operations

```python
# Convert blocking I/O to async
import asyncio

class OptimizedChatHandler:
    def __init__(self):
        self.semaphore = asyncio.Semaphore(100)  # Limit concurrent operations

    async def handle_chat_batch(self, messages: list[str]):
        """Process multiple messages concurrently."""
        tasks = [
            self.handle_single_message(msg)
            for msg in messages
        ]
        return await asyncio.gather(*tasks)

    async def handle_single_message(self, message: str):
        """Single message with concurrency limit."""
        async with self.semaphore:
            return await self.process_message(message)

    async def process_message(self, message: str):
        """Actual processing."""
        # Simulate async operations
        provider = await get_provider()
        response = await provider.complete(message)
        await store_result(response)
        return response
```

### Connection Pooling Across Services

```python
# Reuse connections across async operations
class ServiceClient:
    _session = None

    @classmethod
    async def get_session(cls):
        """Get or create shared session."""
        if cls._session is None:
            import aiohttp
            timeout = aiohttp.ClientTimeout(total=30)
            cls._session = aiohttp.ClientSession(timeout=timeout)
        return cls._session

    @classmethod
    async def close(cls):
        """Close shared session."""
        if cls._session:
            await cls._session.close()

# Usage
async def call_multiple_services():
    session = await ServiceClient.get_session()

    tasks = [
        session.get("https://api1.example.com/data"),
        session.get("https://api2.example.com/data"),
    ]

    results = await asyncio.gather(*tasks)
    return results
```

---

## Monitoring & Alerting

### Performance Baseline

```yaml
# config/performance_baselines.yaml
baselines:
    api_latency_p95_ms:
        development: 100
        staging: 200
        production: 250
        alert_threshold: 500

    db_query_time_ms:
        baseline: 50
        alert_threshold: 200

    cache_hit_rate:
        baseline: 0.85
        alert_threshold: 0.70

    error_rate:
        baseline: 0.001
        alert_threshold: 0.01

    memory_usage_gb:
        single_instance: 1
        alert_threshold: 2
```

### Auto-Scaling Configuration

```python
# Azure auto-scaling rules
class AutoScalingManager:
    def __init__(self, app_service_plan_id: str):
        self.plan_id = app_service_plan_id

    def create_scale_up_rule(self):
        """Scale up when CPU > 70%."""
        return {
            "metric_trigger": {
                "metric_name": "CpuPercentage",
                "statistic": "Average",
                "operator": "GreaterThan",
                "threshold": 70,
                "time_grain": "PT1M",
                "time_window": "PT5M"
            },
            "scale_action": {
                "direction": "Increase",
                "type": "ChangeCount",
                "value": 1,
                "cooldown": "PT5M"
            }
        }

    def create_scale_down_rule(self):
        """Scale down when CPU < 30%."""
        return {
            "metric_trigger": {
                "metric_name": "CpuPercentage",
                "statistic": "Average",
                "operator": "LessThan",
                "threshold": 30,
                "time_grain": "PT1M",
                "time_window": "PT10M"
            },
            "scale_action": {
                "direction": "Decrease",
                "type": "ChangeCount",
                "value": 1,
                "cooldown": "PT5M"
            }
        }
```


## Performance Testing

### Benchmark Suite

```bash
#!/bin/bash
# scripts/run_benchmarks.sh

echo "🏃 Running Aria performance benchmarks..."

# Test 1: Chat API latency
echo "Test 1: Chat API Latency (p95)"
locust -f load_test_aria.py \
  --host=http://localhost:7071 \
  --users=100 \
  --spawn-rate=10 \
  --run-time=5m \
  --headless

# Test 2: Database query performance
echo "Test 2: Database Query Performance"
python -c "
import time
from shared.sql_engine import get_engine
engine = get_engine()
conn = engine.connect()

start = time.time()
for _ in range(1000):
    conn.execute('SELECT 1')
elapsed = (time.time() - start) * 1000
print(f'1000 queries: {elapsed:.0f}ms ({elapsed/1000:.1f}ms per query)')
"

# Test 3: Cache performance
echo "Test 3: Cache Performance"
redis-benchmark -t get,set -n 10000 -q

# Test 4: Memory usage
echo "Test 4: Memory Usage Trend"
python scripts/profile_memory.py

echo "✅ Benchmarks complete"
```

### Regression Testing

```python
# Track performance over time
class PerformanceRegression:
    def __init__(self, baseline_file: str):
        self.baseline = json.load(open(baseline_file))

    def check_regression(self, current_metrics: dict) -> dict:
        """Compare current vs baseline."""
        regressions = {}

        for metric_name, baseline_value in self.baseline.items():
            current_value = current_metrics.get(metric_name)
            if current_value is None:
                continue

            percent_change = ((current_value - baseline_value) / baseline_value) * 100

            # Alert if >10% regression
            if percent_change > 10:
                regressions[metric_name] = {
                    "baseline": baseline_value,
                    "current": current_value,
                    "change_percent": percent_change
                }

        return regressions
```

---

## Scalability Best Practices

### Design Patterns

- [ ] Stateless application servers
- [ ] Distributed caching layer
- [ ] Database read replicas
- [ ] Async message queues
- [ ] CDN for static assets
- [ ] Horizontal load balancing

### Infrastructure

- [ ] Auto-scaling policies configured
- [ ] Multiple availability zones
- [ ] Cross-region replication
- [ ] Connection pooling enabled
- [ ] Rate limiting implemented
- [ ] Circuit breakers for external APIs

### Monitoring

- [ ] Performance baselines tracked
- [ ] Latency alerts configured
- [ ] Capacity utilization monitored
- [ ] Database metrics watched
- [ ] Cache hit rates tracked
- [ ] Auto-scaling events logged

### Testing

- [ ] Load tests run regularly
- [ ] Spike tests (sudden traffic)
- [ ] Soak tests (sustained load)
- [ ] Stress tests (breaking point)
- [ ] Chaos engineering
- [ ] Failover scenarios tested
